In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# Install once if needed:
# pip install datasets

from datasets import load_dataset

# Load the CSV with Hugging Face Datasets (no pandas)
dataset = load_dataset(
    "csv",
    data_files=r"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    split="train"
)

# Create combined_text = prompt + " " + A
def create_combined_text(example):
    return {
        "combined_text": str(example["prompt"]) + " " + str(example["A"])
    }

dataset = dataset.map(create_combined_text)

# Zero-indexed row 51
character_length = len(dataset[51]["combined_text"])

print("combined_text at index 51:")
print(dataset[51]["combined_text"])
print("\nCharacter length:", character_length)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

combined_text at index 51:
Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.

Character length: 614


In [3]:
# Install once if needed:
# pip install transformers

from transformers import AutoTokenizer

# Initialize the BERT base uncased tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Total number of tokens in its vocabulary
vocab_size = tokenizer.vocab_size

print("Tokenizer vocabulary size:", vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer vocabulary size: 30522


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Extract the integer token ID for BERT's separator token
sep_token_id = tokenizer.sep_token_id

print("[SEP] token:", tokenizer.sep_token)
print("[SEP] token ID:", sep_token_id)

[SEP] token: [SEP]
[SEP] token ID: 102


In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

train_dataset = load_dataset(
    "csv",
    data_files=r"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    split="train"
)

def clean_prompt(example):
    return {
        "prompt_clean": "" if example["prompt"] is None else str(example["prompt"])
    }

train_dataset = train_dataset.map(clean_prompt)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Convert Hugging Face Column object into a normal Python list
all_prompts = list(train_dataset["prompt_clean"])

tokenized_prompts = tokenizer(
    all_prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print("input_ids shape:", list(tokenized_prompts["input_ids"].shape))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

input_ids shape: [2000, 128]


In [6]:
# BERT base architecture values
hidden_size = 768
num_attention_heads = 12

# Each attention head receives an equal portion of the hidden size
attention_head_size = hidden_size // num_attention_heads

print("Individual attention-head size:", attention_head_size)

Individual attention-head size: 64


In [7]:
# Install once if needed:
# pip install datasets transformers torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load the training data using Hugging Face Datasets
train_dataset = load_dataset(
    "csv",
    data_files=r"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    split="train"
)

# Load BERT tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Get the prompt at zero-indexed row 0
prompt_row_0 = str(train_dataset[0]["prompt"])

# Default tokenization: no manual padding or truncation
inputs = tokenizer(prompt_row_0, return_tensors="pt")

# Run the prompt through BERT
outputs = model(**inputs)

# Shape: [batch_size, sequence_length, hidden_size]
print("last_hidden_state shape:", list(outputs.last_hidden_state.shape))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: [1, 31, 768]


In [8]:
# `outputs` is the model output from the previous question.

# last_hidden_state shape: [batch_size, sequence_length, hidden_size]
last_hidden_state = outputs.last_hidden_state

# [CLS] is always token index 0 for the first (and only) prompt
cls_embedding = last_hidden_state[0, 0, :]

# Sum the first five values and round to four decimal places
first_five_sum = cls_embedding[:5].sum().item()
rounded_sum = round(first_five_sum, 4)

print("Sum of first 5 [CLS] embedding values:", rounded_sum)

Sum of first 5 [CLS] embedding values: -1.2001


In [9]:
# Install once if needed:
# pip install transformers torch

from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

# Keep the model deterministic for inference
model.eval()

text = "Light-ion fusion is a technique."

# Tokenize the given string
inputs = tokenizer(text, return_tensors="pt")

# Inspect tokens to locate "fusion"
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens:", tokens)

# Find fusion's token index
fusion_index = tokens.index("fusion")
print("fusion token index:", fusion_index)

# Forward pass
with __import__("torch").no_grad():
    outputs = model(**inputs)

# Shape of each attention layer:
# [batch_size, num_heads, sequence_length, sequence_length]
last_layer_attention = outputs.attentions[-1]

# First item in batch, first head, CLS query index 0, fusion key index
cls_to_fusion_attention = last_layer_attention[0, 0, 0, fusion_index].item()

print(
    "[CLS] → fusion attention:",
    round(cls_to_fusion_attention, 4)
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
fusion token index: 4
[CLS] → fusion attention: 0.1025


In [10]:
# Install once if needed:
# pip install datasets sentence-transformers

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

# Load train dataset without pandas
train_dataset = load_dataset(
    "csv",
    data_files=r"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    split="train"
)

# Load the sentence-transformers model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Extract prompt and Option B from zero-indexed row 0
prompt = str(train_dataset[0]["prompt"])
option_b = str(train_dataset[0]["B"])

# Create embeddings using .encode()
prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_b_embedding = model.encode(option_b, convert_to_tensor=True)

# Calculate cosine similarity with sentence-transformers utility
similarity = util.cos_sim(prompt_embedding, option_b_embedding).item()

print("Cosine similarity:", round(similarity, 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine similarity: 0.7658


In [11]:
# Install once if needed:
# pip install datasets scikit-learn sentence-transformers torch

import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
train_dataset = load_dataset(
    "csv",
    data_files=r"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    split="train"
)

OPTION_LABELS = ["A", "B", "C", "D", "E"]
LABEL_COLUMN = "answer"  # Change only if your correct-answer column has another name

prompts = ["" if x is None else str(x) for x in train_dataset["prompt"]]
options = {
    label: ["" if x is None else str(x) for x in train_dataset[label]]
    for label in OPTION_LABELS
}
correct_answers = [str(x).strip().upper() for x in train_dataset[LABEL_COLUMN]]

# ------------------------------------------------------------
# 2. Evaluation helper: MAP@3
# ------------------------------------------------------------
def map_at_3(predictions, actual_labels):
    total_score = 0.0

    for predicted_top3, actual in zip(predictions, actual_labels):
        if actual in predicted_top3:
            # Rank positions are 1, 2, and 3, so scores are 1, 1/2, 1/3.
            rank = predicted_top3.index(actual) + 1
            total_score += 1 / rank

    return total_score / len(actual_labels)

# ------------------------------------------------------------
# 3. Pipeline 1: TF-IDF + cosine similarity
# ------------------------------------------------------------
# Fit TF-IDF using all prompts and all five answer-option texts.
all_text = prompts.copy()
for label in OPTION_LABELS:
    all_text.extend(options[label])

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(all_text)

prompt_tfidf = tfidf_vectorizer.transform(prompts)

tfidf_top3_predictions = []

for row_index in range(len(train_dataset)):
    option_texts = [options[label][row_index] for label in OPTION_LABELS]
    option_tfidf = tfidf_vectorizer.transform(option_texts)

    # Cosine score between this question's prompt and all five options
    scores = cosine_similarity(prompt_tfidf[row_index], option_tfidf)[0]

    # Highest score first; output answer labels rather than numeric positions
    ranked_indices = np.argsort(scores)[::-1]
    top3 = [OPTION_LABELS[index] for index in ranked_indices[:3]]
    tfidf_top3_predictions.append(top3)

tfidf_map3 = map_at_3(tfidf_top3_predictions, correct_answers)

# ------------------------------------------------------------
# 4. Pipeline 2: all-MiniLM-L6-v2 + cosine similarity
# ------------------------------------------------------------
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Embeddings for every prompt
prompt_embeddings = model.encode(
    prompts,
    convert_to_tensor=True,
    show_progress_bar=True
)

# Embeddings for every A/B/C/D/E option
option_embeddings = {}

for label in OPTION_LABELS:
    option_embeddings[label] = model.encode(
        options[label],
        convert_to_tensor=True,
        show_progress_bar=True
    )

minilm_top3_predictions = []

for row_index in range(len(train_dataset)):
    # Get the five option vectors for this question
    current_option_vectors = [
        option_embeddings[label][row_index]
        for label in OPTION_LABELS
    ]

    # util.cos_sim returns five cosine-similarity scores
    scores = util.cos_sim(
        prompt_embeddings[row_index],
        current_option_vectors
    )[0].cpu().numpy()

    ranked_indices = np.argsort(scores)[::-1]
    top3 = [OPTION_LABELS[index] for index in ranked_indices[:3]]
    minilm_top3_predictions.append(top3)

minilm_map3 = map_at_3(minilm_top3_predictions, correct_answers)

# ------------------------------------------------------------
# 5. Count questions improved by MiniLM over TF-IDF
# ------------------------------------------------------------
improved_count = sum(
    (correct not in tfidf_top3) and (correct in minilm_top3)
    for correct, tfidf_top3, minilm_top3 in zip(
        correct_answers,
        tfidf_top3_predictions,
        minilm_top3_predictions
    )
)

# ------------------------------------------------------------
# 6. Results
# ------------------------------------------------------------
print(f"TF-IDF MAP@3: {tfidf_map3:.4f}")
print(f"all-MiniLM-L6-v2 MAP@3: {minilm_map3:.4f}")
print(f"Improved questions (TF-IDF miss, MiniLM hit): {improved_count}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

ValueError: only one element tensors can be converted to Python scalars

In [12]:
# Install once if needed:
# pip install datasets transformers torch

from datasets import load_dataset
from transformers import pipeline

# Load the training dataset
train_dataset = load_dataset(
    "csv",
    data_files=r"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    split="train"
)

# Default model for this pipeline: facebook/bart-large-mnli
zero_shot_classifier = pipeline("zero-shot-classification")

# Use zero-indexed row 1 (the second row)
prompt = str(train_dataset[1]["prompt"])

# Candidate labels are the actual text of Options A, B, and C
candidate_labels = [
    str(train_dataset[1]["A"]),
    str(train_dataset[1]["B"]),
    str(train_dataset[1]["C"])
]

result = zero_shot_classifier(
    prompt,
    candidate_labels=candidate_labels
)

print("Ranked options:", result["labels"])
print("Scores:", result["scores"])
print("Top-ranked probability:", round(result["scores"][0], 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Ranked options: ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult

In [13]:
# This continues from the previous zero-shot-classification code.

# Sum of Softmax-based probabilities from multi_label=False
softmax_probability_sum = sum(result["scores"])

# Run the same classification with independent sigmoid probabilities
multi_label_result = zero_shot_classifier(
    prompt,
    candidate_labels=candidate_labels,
    multi_label=True
)

# Sum of independent sigmoid probabilities
sigmoid_probability_sum = sum(multi_label_result["scores"])

# Absolute difference between the two sums
absolute_difference = abs(softmax_probability_sum - sigmoid_probability_sum)

print("Softmax probability sum:", softmax_probability_sum)
print("Independent sigmoid probability sum:", sigmoid_probability_sum)
print("Absolute difference:", round(absolute_difference, 4))

Softmax probability sum: 0.9999999701976776
Independent sigmoid probability sum: 0.0005095975611766335
Absolute difference: 0.9995


In [14]:
# Install once if needed:
# pip install datasets transformers torch sentencepiece

from datasets import load_dataset
from transformers import pipeline

# Load the training dataset
train_dataset = load_dataset(
    "csv",
    data_files=r"/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
    split="train"
)

# Load FLAN-T5 Small as a text-to-text generation pipeline
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)

# Extract zero-indexed row 0
row = train_dataset[0]

# Construct the required prompt exactly
generation_prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    "Answer with just the letter A or B."
)

# Generate the answer
result = generator(
    generation_prompt,
    max_new_tokens=5
)

generated_answer = result[0]["generated_text"]

pr

config.json: 0.00B [00:00, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"